# Seminar 4: Data Formats and APIs

November 11, 2025

______________________________

## What are we going to learn

- What are the most popular data formats and how to work with them
- Databases, SQL and data manipulation
- HTTP requests
- How to scrape websites 

## 1. Data formats

### Data Serialization and Formats
Serialization is the process of converting data structures or objects into a format that can be easily stored and shared. Common data formats in Python include:
- **JSON**: Lightweight and commonly used for APIs.
- **CSV**: Ideal for tabular data, especially in spreadsheets.
- **Excel**: Classic data storage everyone is aware of.

#### JSON

In [14]:
import json
import pandas as pd
import os
    
data = {
    "name": "Alice",
    "age": 30,
    "is_member": True,
    "hobbies": ["reading", "biking", "coding"]
}

In [15]:
filepath = os.path.join('Data', 'sample.json')
filepath

'Data\\sample.json'

In [16]:
with open(filepath, 'w') as f:
    json.dump(data, f)

#### CSV

In [17]:
import csv

data = [
    ["Name", "Age", "Occupation"],
    ["Alice", 30, "Engineer"],
    ["Bob", 25, "Designer"],
    ["Charlie", 35, "Teacher"]
]


In [18]:
filepath = os.path.join('Data', 'sample.csv')
filepath

'Data\\sample.csv'

In [19]:
with open(filepath, 'w') as f:
    writer = csv.writer(f)
    writer.writerows(data)

In [20]:
filepath = os.path.join('Data', 'sample.csv')
with open(filepath, 'w') as f:
    df = pd.DataFrame(data)
    df.to_csv(f)

#### Excel

In [28]:
data = {
    "name": "Thomas",
    "age": 30,
    "is_member": True,
    "hobbies": "reading, biking, coding"
}

In [ ]:
filepath = os.path.join('Data', 'sample.xlsx')
with open(filepath, 'w') as f:
    df = pd.DataFrame([data])
    df.to_excel(f, index = False)

ModuleNotFoundError: No module named 'openpyxl'

## 2. Requests and web scraping

In [32]:
import requests # for making HTTP requests
import pandas as pd 
import time
import re # Regex = Regular Expressions

Time package small things:

In [33]:
%%time
print("Hello, World!")

Hello, World!
CPU times: total: 0 ns
Wall time: 321 μs


In [34]:
t0 = time.time()
time.sleep(1)
t1 = time.time()
print("Time elapsed: ", t1-t0, " seconds")

Time elapsed:  1.0008103847503662  seconds


In [35]:
%%time 
time.sleep(2)

CPU times: total: 0 ns
Wall time: 2 s


In [36]:
import random

In [39]:
%%time
r_time = random.uniform(0.5, 1.2)
print("Sleeping for ", r_time, " seconds")
time.sleep(r_time)

Sleeping for  0.8489005719340098  seconds
CPU times: total: 0 ns
Wall time: 850 ms


#### Task 1: Requesting API

Let us work with data of sreality.cz which we can access via their api. An intuition is that the api is limited for a number of requests (but not verified).

##### 1a. Create a function requesting data from sreality

```python
base_url = 'https://www.sreality.cz/api/cs/v2/estates?category_main_cb=1&category_type_cb=1&locality_region_id=10&per_page60&page={}'.format(i)

r = requests.get(base_url)
d = r.json()
```

0) function should parametrize: 
    * `category_main_cb` - `{'flat':1, 'house':2, 'land':3 }`
    * `category_type_cb` - `{'sell':1,'rent':2}`
    * `locality_region_id` - use 10 as default value
    * `page` parameter
1) use string inputs for `category_main_cb` and `category_type_cb`
2) include `try/except` clause to handle errors
3) function should return JSON data in python types
4) do not forget to sleep each request at least 0.5s

In [40]:
text_to_format = 'I bought {pnps} pineapples, asnd {apls} apples'
text_to_format.format(pnps = 35, apls = 120)

'I bought 35 pineapples, asnd 120 apples'

In [41]:
def request_sreality(page, category_main_str, category_type_str, locality_region_id=10):
    """
    Request data from sreality.cz API
    :param page: page number
    :param category_main_str: category of the property
    :param category_type_str: type of the offer
    :param locality_region_id: region id
    :return json: json response
    """
    category_main_dict = {'flat':1, 'house':2, 'land':3 }
    category_type_dict = {'sell':1,'rent':2}
    category_main = category_main_dict[category_main_str]
    category_type = category_type_dict[category_type_str]
    
    template_url = 'https://www.sreality.cz/api/cs/v2/estates?category_main_cb={category_main}&category_type_cb={category_type}&locality_region_id={locality_region_id}&per_page60&page={page}'
    url = template_url.format(category_main = category_main, category_type = category_type,
                              locality_region_id = locality_region_id, page = page)
    
    time.sleep(random.uniform(0.2, 5))

    try:
        response = requests.get(url)
    except Exception:
        print('Could not retrive the data')

    return response.json()

In [42]:
d = request_sreality(0, 'flat', 'sell', locality_region_id=10)

Inspect the element `d`:

In [43]:
d.keys()

dict_keys(['meta_description', 'result_size', '_embedded', 'filterLabels', 'title', 'filter', '_links', 'locality', 'locality_dativ', 'logged_in', 'per_page', 'category_instrumental', 'page', 'filterLabels2'])

In [44]:
d['_embedded'].keys()

dict_keys(['estates', 'is_saved', 'not_precise_location_count'])

In [50]:
d['_embedded']['estates'][0]

{'labelsReleased': [['in_construction', 'balcony', 'parking_lots'], ['shop']],
 'has_panorama': 0,
 'labels': ['Ve výstavbě', 'Balkon', 'Parkování', 'Obchod 7 min. pěšky'],
 'is_auction': False,
 'labelsAll': [['in_construction',
   'personal',
   'balcony',
   'cellar',
   'parking_lots'],
  ['small_shop',
   'candy_shop',
   'tavern',
   'vet',
   'playground',
   'theater',
   'natural_attraction',
   'movies',
   'restaurant',
   'sports',
   'bus_public_transport',
   'medic',
   'metro',
   'post_office',
   'tram',
   'drugstore',
   'train',
   'school',
   'shop',
   'kindergarten',
   'atm']],
 'seo': {'category_main_cb': 1,
  'category_sub_cb': 6,
  'category_type_cb': 1,
  'locality': 'praha-vysocany-odkolkova'},
 'exclusively_at_rk': 0,
 'category': 1,
 'has_floor_plan': 1,
 '_embedded': {'favourite': {'is_favourite': False,
   '_links': {'self': {'profile': '/favourite/doc',
     'href': '/cs/v2/favourite/4258747212',
     'title': 'Oblibene inzeraty'}}},
  'note': {'note

In [52]:
print(type(d))

<class 'dict'>


AttributeError: module 'pandas' has no attribute 'head'

### 1b. Create a function converting sreality json data into pandas dataframe

In [58]:
def convert_sreality_data_to_df(sreality_data):
    return pd.DataFrame(sreality_data['_embedded']['estates'])

raw = convert_sreality_data_to_df(d)

### 1c. link function `1b` into function `1a`

### 1c. Combining multiple requests into single df

* Function should parametrize:
    * `start_page` and `end_page`
    * request parameters
* construct a list of individual request dfs
* then feed it into `pd.concat` function

In [59]:
def request_multiple_sreality(start_page, end_page, category_main_str, category_type_str, locality_region_id=10):
    pages = range(start_page, end_page+1)
    df_list = []
    for page in pages:
        df = convert_sreality_data_to_df(request_sreality(page, category_main_str, category_type_str, locality_region_id))
        df_list.append(df)
    res = pd.concat(df_list)
    return res

In [60]:
df = request_multiple_sreality(1, 5, 'flat', 'sell', locality_region_id=10)

In [61]:
df

,labelsReleased,has_panorama,labels,is_auction,labelsAll,seo,exclusively_at_rk,category,has_floor_plan,_embedded,...,hash_id,attractive_offer,price,price_czk,_links,rus,name,region_tip,gps,has_matterport_url
0,"[[in_construction, balcony, parking_lots], [sh...",0,"[Ve výstavbě, Balkon, Parkování, Obchod 7 min....",False,"[[in_construction, personal, balcony, cellar, ...","{'category_main_cb': 1, 'category_sub_cb': 4, ...",0,1,1,"{'favourite': {'is_favourite': False, '_links'...",...,299324236,0,0,"{'value_raw': 0, 'unit': '', 'name': 'Celková ...",{'dynamicDown': [{'href': 'https://d18-a.sdn.c...,False,Prodej bytu 2+kk 91 m²,3564655,"{'lat': 50.098651246491855, 'lon': 14.51975675...",False
1,"[[loggia, parking_lots, garage], []]",0,"[Lodžie, Parkování, Garáž]",False,"[[personal, loggia, brick, parking_lots, garag...","{'category_main_cb': 1, 'category_sub_cb': 11,...",0,1,1,"{'favourite': {'is_favourite': False, '_links'...",...,3758113612,0,35681000,"{'value_raw': 35681000, 'unit': '', 'name': 'C...",{'dynamicDown': [{'href': 'https://d18-a.sdn.c...,False,Prodej bytu 5+1 278 m²,0,"{'lat': 50.03584224649185, 'lon': 14.556567753...",False
2,"[[balcony], []]",0,[Balkon],False,"[[personal, balcony, brick, cellar, elevator, ...","{'category_main_cb': 1, 'category_sub_cb': 8, ...",1,1,1,"{'favourite': {'is_favourite': False, '_links'...",...,208159564,0,20557000,"{'value_raw': 20557000, 'unit': '', 'name': 'C...",{'dynamicDown': [{'href': 'https://d18-a.sdn.c...,False,Prodej bytu 4+kk 110 m²,0,"{'lat': 50.075728246491856, 'lon': 14.51031375...",False
3,"[[], []]",0,[],False,"[[personal, brick, cellar, elevator, partly_fu...","{'category_main_cb': 1, 'category_sub_cb': 4, ...",1,1,1,"{'favourite': {'is_favourite': False, '_links'...",...,219317068,0,22906000,"{'value_raw': 22906000, 'unit': '', 'name': 'C...",{'dynamicDown': [{'href': 'https://d18-a.sdn.c...,False,Prodej bytu 2+kk 93 m²,0,"{'lat': 50.06106424649185, 'lon': 14.460659753...",False
4,"[[], [shop]]",0,[Obchod 5 min. pěšky],False,"[[personal, brick, cellar, partly_furnished], ...","{'category_main_cb': 1, 'category_sub_cb': 4, ...",1,1,1,"{'favourite': {'is_favourite': False, '_links'...",...,346104652,0,12334000,"{'value_raw': 12334000, 'unit': '', 'name': 'C...",{'dynamicDown': [{'href': 'https://d18-a.sdn.c...,False,Prodej bytu 2+kk 77 m²,0,"{'lat': 50.028594246491856, 'lon': 14.46639175...",False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
16,"[[after_reconstruction, loggia, panel], [drugs...",0,"[Po rekonstrukci, Lodžie, Panelová, Lékárna 8 ...",False,"[[personal, after_reconstruction, loggia, pane...","{'category_main_cb': 1, 'category_sub_cb': 8, ...",1,1,1,"{'favourite': {'is_favourite': False, '_links'...",...,1057899340,0,17616000,"{'value_raw': 17616000, 'unit': '', 'name': 'C...",{'dynamicDown': [{'href': 'https://d18-a.sdn.c...,False,Prodej bytu 4+kk 118 m²,0,"{'lat': 50.07784924649186, 'lon': 14.507122753...",False
17,"[[furnished], [post_office, drugstore]]",0,"[Vybavený, Pošta 1 min. pěšky, Lékárna 1 min. ...",False,"[[personal, terrace, brick, parking_lots, gara...","{'category_main_cb': 1, 'category_sub_cb': 10,...",1,1,1,"{'favourite': {'is_favourite': False, '_links'...",...,1270645580,0,35226000,"{'value_raw': 35226000, 'unit': '', 'name': 'C...",{'dynamicDown': [{'href': 'https://d18-a.sdn.c...,False,Prodej bytu 5+kk 248 m²,0,"{'lat': 50.021517246491854, 'lon': 14.35820875...",False
18,"[[], [metro]]",0,[Metro 3 min. pěšky],False,"[[personal, terrace, cellar, parking_lots, gar...","{'category_main_cb': 1, 'category_sub_cb': 8, ...",1,1,0,"{'favourite': {'is_favourite': False, '_links'...",...,3176285004,0,0,"{'value_raw': 0, 'unit': '', 'name': 'Celková ...",{'dynamicDown': [{'href': 'https://d18-a.sdn.c...,False,Prodej bytu 4+kk 176 m² (Jednopodlažní),0,"{'lat': 50.079364246491856, 'lon': 14.46824975...",False
19,"[[partly_furnished], [metro, post_office]]",0,"[Částečně vybavený, Metro 4 min. pěšky, Pošta

In [68]:
# reset the index


#### Task 2: Cleaning data

__2a. Filter columns__
* filter only columns: `['locality', 'price', 'name', 'gps','hash_id','exclusively_at_rk']`
* use `.copy()` to avoid `SettingWithCopyWarning` later


In [70]:
# display the columns
columns_to_display = ['locality', 'price', 'name', 'gps','hash_id','exclusively_at_rk']
df_clean = df[columns_to_display].copy()
df_clean.head()

,locality,price,name,gps,hash_id,exclusively_at_rk
0,Praha 9 - Vysočany,0,Prodej bytu 2+kk 91 m²,"{'lat': 50.098651246491855, 'lon': 14.51975675...",299324236,0
1,Praha 10 - Hostivař,35681000,Prodej bytu 5+1 278 m²,"{'lat': 50.03584224649185, 'lon': 14.556567753...",3758113612,0
2,Praha 3 - Žižkov,20557000,Prodej bytu 4+kk 110 m²,"{'lat': 50.075728246491856, 'lon': 14.51031375...",208159564,1
3,Praha 2 - Vinohrady,22906000,Prodej bytu 2+kk 93 m²,"{'lat': 50.06106424649185, 'lon': 14.460659753...",219317068,1
4,Praha 4 - Krč,12334000,Prodej bytu 2+kk 77 m²,"{'lat': 50.028594246491856, 'lon': 14.46639175...",346104652,1


Copy selection into new one

### 2b: GPS
* Convert dictionary in `gps` column into two columns - `lat` and `lon`
* use apply function on gps column
* Note apply can return multiple columns

In [69]:
df_clean[["lag", 'lon']] = df_clean.gps.apply(lambda x: pd.Series(x))

AttributeError: 'DataFrame' object has no attribute 'gps'

In [72]:
df_clean.head()

,locality,price,name,gps,hash_id,exclusively_at_rk
0,Praha 9 - Vysočany,0,Prodej bytu 2+kk 91 m²,"{'lat': 50.098651246491855, 'lon': 14.51975675...",299324236,0
1,Praha 10 - Hostivař,35681000,Prodej bytu 5+1 278 m²,"{'lat': 50.03584224649185, 'lon': 14.556567753...",3758113612,0
2,Praha 3 - Žižkov,20557000,Prodej bytu 4+kk 110 m²,"{'lat': 50.075728246491856, 'lon': 14.51031375...",208159564,1
3,Praha 2 - Vinohrady,22906000,Prodej bytu 2+kk 93 m²,"{'lat': 50.06106424649185, 'lon': 14.460659753...",219317068,1
4,Praha 4 - Krč,12334000,Prodej bytu 2+kk 77 m²,"{'lat': 50.028594246491856, 'lon': 14.46639175...",346104652,1


In [ ]:
df_clean.drop(columns='gps', inplace=True)

KeyError: "['gps'] not found in axis"

### 2c. Get flat type from name

* Name is always represented by string `Prodej bytu [type of flat] [Area] m^2`
* Try picking third word in string
* Check meaningfulness using `.value_counts()`

In [103]:
# type of flat (' ')


In [96]:
# flat_type ()



### 2d. Get the area of a flat from name

* Naive: select the word before last word - save as `area1`
* Then try navigating using the index of `'m²'` in the splitted string - save as `area2`
* If this also fail, then you will need to use regex - `import re` - save as `area3`

In [ ]:
def name_to_area(nm):


In [120]:
# regular expression to find characters before '+' and behind '+'
def flat_type(x:str):
    # \d+ represents one or more digit(s)
    # \s is a space character
    # \w+ is one or more word character(s)
    # * represents 0 or more


__Check if the three areas are equal__